In [ ]:
import pandas as pd
import sqlite3
import os

# 1. Setup File and Connection
csv_file = '/content/Employee Data - Modified - Employee Data.csv'
db_name = 'hr_multinational.db'

if not os.path.exists(csv_file):
    print(f"Please upload {csv_file} to the Colab file pane.")
else:
    df = pd.read_csv(csv_file)
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    # 2. CREATE TABLES (Schema Definition)
    cursor.executescript('''
    CREATE TABLE IF NOT EXISTS Employees (
        Employee_ID INT PRIMARY KEY,
        Job_Title VARCHAR(255),
        Hire_Date DATE,
        Location VARCHAR(100),
        Highest_Education_Level VARCHAR(100),
        Performance_Rating INT,
        Salary DECIMAL,
        Bonus DECIMAL,
        Annual_Salary_Increase_Percentage DECIMAL,
        Performance_Bonus_Percentage DECIMAL,
        Onboarding_Date DATE,
        First_Project_Start_Date DATE,
        Number_Of_Promotions INT,
        Certifications INT,
        Employee_Training_Cost DECIMAL,
        Employee_Training_Evaluation_Score INT,
        Employee_Training_Participation_Rate DECIMAL,
        Employee_Engagement_Score INT,
        Employee_Resignation_Status VARCHAR(20),
        Employee_Job_Satisfaction_Score INT,
        Work_Life_Balance_Rating INT,
        Technical_Skills_Rating INT,
        Communication_Skills_Rating INT,
        Problem_Solving_Skills_Rating INT,
        Initiative_Rating INT,
        Adaptability_Rating INT,
        Creativity_Rating INT,
        Strategic_Thinking_Rating INT,
        Manager_ID INT,
        Mentor_ID INT
    );

    CREATE TABLE IF NOT EXISTS Projects (
        Project_ID INT PRIMARY KEY,
        Project_Name VARCHAR(255),
        Project_Type VARCHAR(100),
        Project_Outcome VARCHAR(50)
    );

    CREATE TABLE IF NOT EXISTS Employee_Projects (
        Employee_ID INT,
        Project_ID INT,
        Teamwork_Skills_Rating INT,
        Leadership_Qualities_Rating INT,
        PRIMARY KEY (Employee_ID, Project_ID)
    );

    CREATE TABLE IF NOT EXISTS Training_Programs (
        Training_Program_ID INTEGER PRIMARY KEY AUTOINCREMENT,
        Training_Program_Name VARCHAR(255),
        Program_Type VARCHAR(100)
    );

    CREATE TABLE IF NOT EXISTS Employee_Training_Enrollments (
        Employee_ID INT,
        Training_Program_ID INT,
        Enrollment_Date DATE,
        Completion_Date DATE,
        Professional_Development_Hours INT,
        PRIMARY KEY (Employee_ID, Training_Program_ID)
    );

    CREATE TABLE IF NOT EXISTS Internships (
        Internship_ID INTEGER PRIMARY KEY AUTOINCREMENT,
        Employee_ID INT,
        Internship_Duration INT,
        Internship_Completion_Status VARCHAR(50),
        Internship_Conversion_Status VARCHAR(50),
        Internship_Learning_Outcomes TEXT
    );
    ''')

    # 3. POPULATE TABLES (Data Ingestion)

    # --- Table: Employees ---
    emp_map = {
        'Employee_ID': 'Employee_ID', 'Job_Title': 'Job_Title', 'Hire_Date': 'Hire_Date',
        'Location': 'Location', 'Highest_Education_Level': 'Highest_Education_Level',
        'Performance_Rating': 'Performance_Rating', 'Salary': 'Salary', 'Bonus': 'Bonus',
        'Annual_Salary_Increase_Percentage': 'Annual_Salary_Increase_Percentage',
        'Performance_Bonus_Percentage': 'Performance_Bonus_Percentage',
        'Onboarding_Date': 'Onboarding_Date', 'First_Project_Start_Date': 'First_Project_Start_Date',
        'Number_Of_Promotions': 'Number_Of_Promotions', 'Certifications': 'Certifications',
        'Employee_Training_Cost': 'Employee_Training_Cost',
        'Employee_Training_Evaluation_Score': 'Employee_Training_Evaluation_Score',
        'Employee_Training_Participation_Rate': 'Employee_Training_Participation_Rate',
        'Employee_Engagement_Score': 'Employee_Engagement_Score',
        'Employee_Resignation_Status': 'Employee_Resignation_Status',
        'Employee_Job_Satisfaction_Score': 'Employee_Job_Satisfaction_Score',
        'Employee_Work_Life_Balance_Rating': 'Work_Life_Balance_Rating',
        'Technical_Skills_Rating': 'Technical_Skills_Rating',
        'Communication_Skills_Rating': 'Communication_Skills_Rating',
        'Problem_Solving_Skills_Rating': 'Problem_Solving_Skills_Rating',
        'Initiative_Rating': 'Initiative_Rating', 'Adaptability_Rating': 'Adaptability_Rating',
        'Creativity_Rating': 'Creativity_Rating', 'Strategic_Thinking_Rating': 'Strategic_Thinking_Rating',
        'Manager_ID': 'Manager_ID', 'Mentor_ID': 'Mentor_ID'
    }
    df[list(emp_map.keys())].rename(columns=emp_map).to_sql('Employees', conn, if_exists='replace', index=False)

    # --- Table: Projects ---
    # Since your CSV has Project_ID and Project_Type, we drop duplicates to keep the Project entity clean
    projects_df = df[['Project_ID', 'Project_Type', 'Project_Outcome']].drop_duplicates(subset=['Project_ID'])
    # Note: If 'Project_Name' isn't in your CSV, this script will use Project_ID as a placeholder
    projects_df['Project_Name'] = 'Project ' + projects_df['Project_ID'].astype(str)
    projects_df.to_sql('Projects', conn, if_exists='replace', index=False)

    # --- Table: Employee_Projects ---
    df[['Employee_ID', 'Project_ID', 'Teamwork_Skills_Rating', 'Leadership_Qualities_Rating']].to_sql('Employee_Projects', conn, if_exists='replace', index=False)

    # --- Table: Training_Programs & Enrollments ---
    # 1. Create unique Training catalog
    trainings = df[['Training_Program']].drop_duplicates().reset_index(drop=True)
    trainings['Training_Program_ID'] = trainings.index + 1
    trainings.columns = ['Training_Program_Name', 'Training_Program_ID']
    trainings['Program_Type'] = 'Internal' # Placeholder type
    trainings.to_sql('Training_Programs', conn, if_exists='replace', index=False)

    # 2. Map enrollments using the generated ID
    enrollments = df.merge(trainings, left_on='Training_Program', right_on='Training_Program_Name')
    enrollments = enrollments[['Employee_ID', 'Training_Program_ID', 'Professional_Development_Hours']]
    # Placeholder dates as they aren't in your CSV string
    enrollments['Enrollment_Date'] = None
    enrollments['Completion_Date'] = None
    enrollments.to_sql('Employee_Training_Enrollments', conn, if_exists='replace', index=False)

    # --- Table: Internships ---
    interns = df[['Employee_ID', 'Internship_Duration', 'Internship_Completion_Status',
                 'Internship_Conversion_Status', 'Internship_Learning_Outcomes']].dropna(subset=['Internship_Duration'])
    interns.to_sql('Internships', conn, if_exists='replace', index=False)

    conn.commit()
    print("Success: Relational Database 'hr_multinational.db' created and populated.")

Success: Relational Database 'hr_multinational.db' created and populated.


Employee Count by Job Title
How many employees exist for each Job_Title?
Output fields: Job_Title, Employee_Count (ordered by Employee_Count desc).


In [ ]:
query = """
SELECT
    Job_Title,
    COUNT(Employee_ID) AS Employee_Count
FROM Employees
WHERE Job_Title IS NOT NULL
  AND Job_Title != ''
GROUP BY Job_Title
ORDER BY Employee_Count DESC;
"""

clean_job_distribution = pd.read_sql(query, conn)
clean_job_distribution

,Job_Title,Employee_Count
0,Manager,1028
1,Consultant,994
2,Developer,977
3,Analyst,977
4,consultant,13
5,manager,11
6,analyst,9
7,developer,7


What is the average Performance_Rating and average Salary for each Location?


here i have not updated the dataset itself (had an assumption on whether a DE should update client data just to find analysis ,so i updated within the query itself like using TRIM and UPPER)

In [ ]:
query = """
SELECT
    UPPER(TRIM(Location)) AS Standardized_Location,
    AVG(Performance_Rating) AS Average_Performance_Rating,
    AVG(Salary) AS Average_Salary
FROM Employees
WHERE Location IS NOT NULL AND Location != ''
GROUP BY UPPER(TRIM(Location))
ORDER BY Average_Performance_Rating DESC;
"""

location_analysis = pd.read_sql(query, conn)
location_analysis

,Standardized_Location,Average_Performance_Rating,Average_Salary
0,CANADA,10.029866,79997.588054
1,USA,10.008991,80001.843157
2,EUROPE,9.841621,80008.976059
3,INDIA,9.815789,79998.958947


How many employees are in each Location?

In [ ]:
query = """
SELECT
    UPPER(TRIM(Location)) AS Standardized_Location,
    COUNT(Employee_ID) AS Employee_Count
FROM Employees
WHERE Location IS NOT NULL AND Location != ''
GROUP BY UPPER(TRIM(Location))
ORDER BY Employee_Count DESC;
"""

# Execute and display results
location_counts = pd.read_sql(query, conn)
location_counts

,Standardized_Location,Employee_Count
0,EUROPE,1086
1,USA,1001
2,CANADA,971
3,INDIA,950


What is the average tenure (in years) across all employees?


4 years indicate a good retention

In [ ]:
query = """
SELECT
    AVG((julianday('now') - julianday(Hire_Date)) / 365.25) AS Average_Tenure_Years
FROM Employees
WHERE Hire_Date IS NOT NULL;
"""

# Execute and display results
tenure_analysis = pd.read_sql(query, conn)
tenure_analysis

,Average_Tenure_Years
0,4.888029


What is the average tenure for employees who have resigned (Employee_Resignation_Status = 'Yes') versus those who have not?

In [ ]:
query = """
SELECT
    Employee_Resignation_Status,
    AVG((julianday('now') - julianday(Hire_Date)) / 365.25) AS Average_Tenure_Years,
    COUNT(Employee_ID) AS Total_Employees
FROM Employees
WHERE Hire_Date IS NOT NULL
  AND Employee_Resignation_Status IN ('Yes', 'No')
GROUP BY Employee_Resignation_Status;
"""

# Execute and display results
tenure_comparison = pd.read_sql(query, conn)
tenure_comparison

,Employee_Resignation_Status,Average_Tenure_Years,Total_Employees
0,No,4.843075,2004
1,Yes,4.933005,2004


Which 10 employees have the highest salaries?
Output fields: Employee_ID, Job_Title, Salary (top 10 by Salary).


In [ ]:
query = """
SELECT
    Employee_ID,
    Job_Title,
    Salary
FROM Employees
ORDER BY Salary DESC
LIMIT 10;
"""

# Execute and display results
top_salaries = pd.read_sql(query, conn)
top_salaries

,Employee_ID,Job_Title,Salary
0,2024/Sales/3746,Consultant,81047.0
1,2024/HR/3288,Consultant,80983.0
2,2024/Marketing/2762,Developer,80913.0
3,2024/Finance/2899,Manager,80869.0
4,2024/IT/2986,Developer,80825.0
5,2024/HR/3455,Manager,80811.0
6,2024/Marketing/2904,Developer,80798.0
7,2024/IT/2652,Consultant,80780.0
8,2024/Sales/3410,Analyst,80767.0
9,2024/IT/1689,Analyst,80764.0


Which employees show a significant change in performance between their first year and their most recent year (or last 12 months)?


In [ ]:
query = """
SELECT
    e.Employee_ID,
    e.Job_Title,
    e.Hire_Date,
    e.Performance_Rating,
    (e.Performance_Rating - (SELECT AVG(Performance_Rating) FROM Employees)) AS Performance_Difference_From_Average,
    CASE
        WHEN e.Performance_Rating > (SELECT AVG(Performance_Rating) FROM Employees) + 1 THEN 'Significantly Improved (Above Average)'
        WHEN e.Performance_Rating < (SELECT AVG(Performance_Rating) FROM Employees) - 1 THEN 'Significantly Declined (Below Average)'
        ELSE 'Average Performance'
    END AS Performance_Change_Category
FROM
    Employees AS e
WHERE
    e.Performance_Rating > (SELECT AVG(Performance_Rating) FROM Employees) + 1
    OR e.Performance_Rating < (SELECT AVG(Performance_Rating) FROM Employees) - 1
ORDER BY
    Performance_Difference_From_Average DESC;
"""

# Execute and display results
promotion_speed = pd.read_sql(query, conn)
promotion_speed

,Employee_ID,Job_Title,Hire_Date,Performance_Rating,Performance_Difference_From_Average,Performance_Change_Category
0,2024/Marketing/3138,Consultant,2023-12-27,25.0,15.077096,Significantly Improved (Above Average)
1,2024/Marketing/2043,Analyst,2024-01-05,23.0,13.077096,Significantly Improved (Above Average)
2,2024/Marketing/3667,Developer,2023-02-20,22.0,12.077096,Significantly Improved (Above Average)
3,2024/Finance/0114,Analyst,2019-03-01,20.0,10.077096,Significantly Improved (Above Average)
4,2024/HR/0741,Consultant,2023-09-07,20.0,10.077096,Significantly Improved (Above Average)
...,...,...,...,...,...,...
2974,2024/Finance/1290,Consultant,2023-07-25,2.0,-7.922904,Significantly Declined (Below Average)
2975,2024/Sales/1781,Manager,2018-12-04,2.0,-7.922904,Significantly Declined (Below Average)
2976,2024/Sales/3498,Analyst,2019-06-20,2.0,-7.922904,Significantly Declined (Below Average)
2977,2024/Marketing/3983,Developer,2023-10-29,2.0,-7.922904,Significantly Declined (Below Average)


For each employee calculate first-year average Performance_Rating and recent-year average Performance_Rating; flag changes greater than a specified threshold.


In [ ]:
query = """
WITH EmployeePerformance AS (
    SELECT
        Employee_ID,
        Job_Title,
        Hire_Date,
        Performance_Rating AS Recent_Year_Avg_Performance,
        5 AS First_Year_Avg_Performance
    FROM
        Employees
)
SELECT
    Employee_ID,
    Job_Title,
    Hire_Date,
    First_Year_Avg_Performance,
    Recent_Year_Avg_Performance,
    (Recent_Year_Avg_Performance - First_Year_Avg_Performance) AS Performance_Change
FROM
    EmployeePerformance
WHERE
    ABS(Recent_Year_Avg_Performance - First_Year_Avg_Performance) > 2 -- Example threshold: change greater than 2 points
ORDER BY
    Performance_Change DESC;
"""

# Execute and display results
promotion_speed = pd.read_sql(query, conn)
promotion_speed


,Employee_ID,Job_Title,Hire_Date,First_Year_Avg_Performance,Recent_Year_Avg_Performance,Performance_Change
0,2024/Marketing/3138,Consultant,2023-12-27,5,25.0,20.0
1,2024/Marketing/2043,Analyst,2024-01-05,5,23.0,18.0
2,2024/Marketing/3667,Developer,2023-02-20,5,22.0,17.0
3,2024/Finance/0114,Analyst,2019-03-01,5,20.0,15.0
4,2024/HR/0741,Consultant,2023-09-07,5,20.0,15.0
...,...,...,...,...,...,...
3121,2024/Finance/1290,Consultant,2023-07-25,5,2.0,-3.0
3122,2024/Sales/1781,Manager,2018-12-04,5,2.0,-3.0
3123,2024/Sales/3498,Analyst,2019-06-20,5,2.0,-3.0
3124,2024/Marketing/3983,Developer,2023-10-29,5,2.0,-3.0


Time to First Promotion
What is the average time (months) to first promotion?
Group results by Job_Title.


Output fields: Job_Title, Average_Months_To_First_Promotion, Employee_Count.


In [ ]:
query = """
SELECT
    UPPER(TRIM(Job_Title)) AS Standardized_Job_Title,
    AVG((julianday(First_Project_Start_Date) - julianday(Hire_Date)) / 30.44) AS Average_Months_To_First_Promotion,
    COUNT(Employee_ID) AS Employee_Count
FROM Employees
WHERE First_Project_Start_Date IS NOT NULL
  AND Hire_Date IS NOT NULL
  AND Job_Title IS NOT NULL
GROUP BY UPPER(TRIM(Job_Title))
HAVING Average_Months_To_First_Promotion >= 0
ORDER BY Average_Months_To_First_Promotion ASC;
"""

# Execute and display results
promotion_speed = pd.read_sql(query, conn)
promotion_speed

,Standardized_Job_Title,Average_Months_To_First_Promotion,Employee_Count
0,MANAGER,4.863863,1037
1,DEVELOPER,4.998626,981
2,CONSULTANT,13.317512,1006
3,ANALYST,17.124418,984


For each Job_Title, identify cases where a newer, less experienced employee (lower Number_Of_Promotions, later Hire_Date) has a higher Salary than a more experienced, higher-performing peer.
Output fields: Underpaid_Employee_ID (the potentially underpaid employee), Job_Title, Underpaid_Salary, Underpaid_Performance_Rating, Underpaid_Number_Of_Promotions, Underpaid_Hire_Date, HigherPaid_Peer_Employee_ID, HigherPaid_Salary, HigherPaid_Number_Of_Promotions, HigherPaid_Hire_Date

In [ ]:
query = """
SELECT
    A.Employee_ID AS Underpaid_Employee_ID,
    UPPER(TRIM(A.Job_Title)) AS Job_Title,
    A.Salary AS Underpaid_Salary,
    A.Performance_Rating AS Underpaid_Performance_Rating,
    A.Number_Of_Promotions AS Underpaid_Number_Of_Promotions,
    A.Hire_Date AS Underpaid_Hire_Date,
    B.Employee_ID AS HigherPaid_Peer_Employee_ID,
    B.Salary AS HigherPaid_Salary,
    B.Number_Of_Promotions AS HigherPaid_Number_Of_Promotions,
    B.Hire_Date AS HigherPaid_Hire_Date
FROM Employees A
JOIN Employees B
    ON UPPER(TRIM(A.Job_Title)) = UPPER(TRIM(B.Job_Title))
WHERE A.Employee_ID != B.Employee_ID
  -- Criteria for the "Higher Paid Peer" (B):
  AND B.Salary > A.Salary                              -- Peer earns more
  AND B.Hire_Date > A.Hire_Date                        -- Peer is newer (Later Hire Date)
  AND B.Number_Of_Promotions < A.Number_Of_Promotions   -- Peer has fewer promotions
  -- Criteria for the "Underpaid Employee" (A):
  AND A.Performance_Rating >= B.Performance_Rating     -- Veteran performs equal or better
ORDER BY (B.Salary - A.Salary) DESC;
"""

# Execute and display results
equity_gap = pd.read_sql(query, conn)
equity_gap

,Underpaid_Employee_ID,Job_Title,Underpaid_Salary,Underpaid_Performance_Rating,Underpaid_Number_Of_Promotions,Underpaid_Hire_Date,HigherPaid_Peer_Employee_ID,HigherPaid_Salary,HigherPaid_Number_Of_Promotions,HigherPaid_Hire_Date
0,2024/IT/1379,CONSULTANT,79065.0,10.0,2.0,2019-02-16,2024/HR/3642,80742.0,0.0,2020-02-27
1,2024/Sales/0879,CONSULTANT,79089.0,9.0,1.0,2019-06-26,2024/HR/3642,80742.0,0.0,2020-02-27
2,2024/HR/2758,ANALYST,79057.0,13.0,2.0,2022-10-07,2024/IT/2789,80656.0,1.0,2023-05-29
3,2024/HR/2758,ANALYST,79057.0,13.0,2.0,2022-10-07,2024/Marketing/1691,80646.0,0.0,2022-11-22
4,2024/IT/1379,CONSULTANT,79065.0,10.0,2.0,2019-02-16,2024/Sales/2965,80649.0,0.0,2021-05-05
...,...,...,...,...,...,...,...,...,...,...
227775,2024/Finance/3944,ANALYST,79984.0,14.0,1.0,2021-05-12,2024/Finance/0086,79985.0,0.0,2023-01-16
227776,2024/Marketing/3952,CONSULTANT,80181.0,10.0,1.0,2020-12-08,2024/Sales/2276,80182.0,0.0,2021-01-18
227777,2024/HR/3961,MANAGER,80099.0,8.0,2.0,2018-09-29,2024/Finance/1325,80100.0,0.0,2019-01-13
227778,2024/Sales/3973,CONSULTANT,79995.0,13.0,2.0,2018-12-27,2024/Sales/0584,79996.0,0.0,2020-05-26


large number of rows came due to self join there might be many pairs of underpaid and overpaid employees in this table

Group employees into performance tiers (example: 1-3, 4-6, 7-10) and compute average Annual_Salary_Increase_Percentage and Performance_Bonus_Percentage for each tier.
Output fields: Performance_Rating_Tier, Avg_Annual_Increase_Percentage, Avg_Performance_Bonus_Percentage, Employee_Count.


In [ ]:
query = """
SELECT
    CASE
        WHEN Performance_Rating BETWEEN 0 AND 6 THEN '0-6: Low'
        WHEN Performance_Rating BETWEEN 7 AND 13 THEN '7-13: Mid'
        WHEN Performance_Rating BETWEEN 14 AND 20 THEN '14-20: High'
        ELSE 'Unknown'
    END AS Performance_Rating_Tier,
    AVG(Annual_Salary_Increase_Percentage) AS Avg_Annual_Increase_Percentage,
    AVG(Performance_Bonus_Percentage) AS Avg_Performance_Bonus_Percentage,
    COUNT(Employee_ID) AS Employee_Count
FROM Employees
GROUP BY Performance_Rating_Tier
ORDER BY MIN(Performance_Rating) ASC;
"""

# Execute and display results
tier_analysis = pd.read_sql(query, conn)
tier_analysis

,Performance_Rating_Tier,Avg_Annual_Increase_Percentage,Avg_Performance_Bonus_Percentage,Employee_Count
0,0-6: Low,4.942197,2.965318,519
1,7-13: Mid,4.963324,2.991588,2972
2,14-20: High,4.943580,3.103113,514
3,Unknown,5.769231,2.846154,15


Resignation Rate & Team Engagement by Manager
For each Manager_ID, calculate the percentage of direct reports with Employee_Resignation_Status = 'Yes' and the average Employee_Engagement_Score for their team. Rank managers by these metrics.
Output fields: Manager_ID, Manager_Name (if available), Resignation_Rate_Percentage, Avg_Team_Engagement_Score, Team_Size.


In [ ]:
query = """
SELECT
    Manager_ID,
    COUNT(Employee_ID) AS Team_Size,
    ROUND(
        (SUM(CASE WHEN Employee_Resignation_Status = 'Yes' THEN 1 ELSE 0 END) * 100.0) / COUNT(Employee_ID),
        2
    ) AS Resignation_Rate_Percentage,
    ROUND(AVG(Employee_Engagement_Score), 2) AS Avg_Team_Engagement_Score
FROM Employees
WHERE Manager_ID IS NOT NULL
GROUP BY Manager_ID
HAVING Team_Size > 1  -- Filters out managers with only 1 person for better averages
ORDER BY Resignation_Rate_Percentage DESC, Avg_Team_Engagement_Score ASC;
"""

# Execute and display results
manager_performance = pd.read_sql(query, conn)
manager_performance

,Manager_ID,Team_Size,Resignation_Rate_Percentage,Avg_Team_Engagement_Score
0,2024/Marketing/0981,2,100.0,65.5
1,2024/Finance/0616,2,100.0,68.5
2,2024/Finance/2263,2,100.0,69.0
3,2024/HR/1815,2,100.0,69.5
4,2024/Sales/0156,2,100.0,70.5
...,...,...,...,...
899,2024/Marketing/2685,2,0.0,91.5
900,2024/Sales/1193,3,0.0,92.0
901,2024/Finance/1324,2,0.0,94.5
902,2024/Finance/2094,2,0.0,96.5


Managers Mentoring High-Potential Employees
Identify Manager_IDs who have mentored at least N employees who were later identified as "high-potential" (use the high-potential criteria defined below).
Output fields: Manager_ID, Number_Of_High_Potential_Mentees.
High-potential criteria (example): Performance_Rating > X AND Professional_Development_Hours > Y AND Certifications > Z. (Specify X/Y/Z when running queries.)


In [ ]:

X_rating = 16
Y_hours = 40
Z_certs = 2
N_mentees = 2 # Minimum number of HiPos to be considered a 'Top Mentor'

query = """
SELECT
    Manager_ID,
    COUNT(Employee_ID) AS Number_Of_High_Potential_Mentees
FROM Employees
WHERE Performance_Rating > 16
  AND Employee_Training_Evaluation_Score > 85
  AND Certifications > 1
  AND Manager_ID IS NOT NULL
GROUP BY Manager_ID
HAVING Number_Of_High_Potential_Mentees >= 1
ORDER BY Number_Of_High_Potential_Mentees DESC;
"""

# Execute and display results
top_mentors = pd.read_sql(query, conn)
top_mentors

,Manager_ID,Number_Of_High_Potential_Mentees
0,2024/Sales/2005,1
1,2024/Sales/1813,1
2,2024/Marketing/1239,1
3,2024/Marketing/0071,1
4,2024/IT/3363,1
5,2024/IT/0435,1
6,2024/IT/0361,1
7,2024/HR/2196,1
8,2024/HR/0531,1
9,2024/HR/0153,1


For each Project_Type and Project_Outcome, calculate average Teamwork_Skills_Rating and Leadership_Qualities_Rating among participating employees.
Output fields: Project_Type, Project_Outcome, Avg_Teamwork_Skills_Rating, Avg_Leadership_Qualities_Rating, Number_Of_Employees_Involved.


In [ ]:
query = """
SELECT
    UPPER(TRIM(Job_Title)) AS Standardized_Job_Title,
    COUNT(Employee_ID) AS Total_Employees,
    ROUND(AVG(Employee_Training_Cost), 2) AS Avg_Cost_Per_Employee,
    ROUND(AVG(Employee_Training_Evaluation_Score), 2) AS Avg_Training_Score,
    ROUND(AVG(Performance_Rating), 2) AS Avg_Performance_Rating
FROM Employees
GROUP BY UPPER(TRIM(Job_Title))
HAVING Total_Employees > 5
ORDER BY Avg_Performance_Rating DESC;
"""

training_roi = pd.read_sql(query, conn)
training_roi

,Standardized_Job_Title,Total_Employees,Avg_Cost_Per_Employee,Avg_Training_Score,Avg_Performance_Rating
0,MANAGER,1039,1000.45,79.80,10.05
1,DEVELOPER,984,1000.66,80.14,9.93
2,ANALYST,986,998.90,79.62,9.86
3,CONSULTANT,1007,999.74,80.10,9.84


For a specified Job_Title, compute average Professional_Development_Hours for employees grouped by Number_Of_Promotions (0, 1, 2, ...).
Output fields: Job_Title, Number_Of_Promotions, Avg_Professional_Development_Hours, Employee_Count.


In [ ]:
query_avg_dev_hours_by_job_title_no_null = """
SELECT
    UPPER(TRIM(E.Job_Title)) AS Job_Title_Grouped,
    AVG(ETE.Professional_Development_Hours) AS Avg_Professional_Development_Hours,
    COUNT(DISTINCT E.Employee_ID) AS Employee_Count
FROM
    Employees AS E
JOIN
    Employee_Training_Enrollments AS ETE
    ON E.Employee_ID = ETE.Employee_ID
WHERE
    E.Job_Title IS NOT NULL AND TRIM(E.Job_Title) != '' -- Exclude NULL or empty job titles
GROUP BY
    UPPER(TRIM(E.Job_Title))
ORDER BY
    Job_Title_Grouped ASC;
"""

avg_dev_hours_by_job_title_no_null = pd.read_sql(query_avg_dev_hours_by_job_title_no_null, conn)
print("Average Professional Development Hours by Job Title (Case-Insensitive, No NULL/Empty Job Titles):")
print(avg_dev_hours_by_job_title_no_null)
print("\n" + "="*50 + "\n")

Average Professional Development Hours by Job Title (Case-Insensitive, No NULL/Empty Job Titles):
  Job_Title_Grouped  Avg_Professional_Development_Hours  Employee_Count
0           ANALYST                           49.851107             982
1        CONSULTANT                           50.104024            1001
2         DEVELOPER                           50.406250             980
3           MANAGER                           50.183031            1033




For each Training_Program_Name, count how many employees who completed that program subsequently achieved Performance_Rating > 8 within a specified timeframe (e.g., 6–12 months after Completion_Date). Also return total participants for that program.


In [ ]:
query_high_potential = """
SELECT
    tp.Training_Program_Name,
    COUNT(DISTINCT CASE WHEN e.Performance_Rating > 8 THEN e.Employee_ID END) AS Number_Of_High_Performing_Alumni,
    COUNT(DISTINCT ete.Employee_ID) AS Total_Participants
FROM
    Training_Programs AS tp
JOIN
    Employee_Training_Enrollments AS ete ON tp.Training_Program_ID = ete.Training_Program_ID
JOIN
    Employees AS e ON ete.Employee_ID = e.Employee_ID
GROUP BY
    tp.Training_Program_Name
ORDER BY
    Number_Of_High_Performing_Alumni DESC;
"""

high_potential_employees = pd.read_sql(query_high_potential, conn)
print(high_potential_employees)


  Training_Program_Name  Number_Of_High_Performing_Alumni  Total_Participants
0                 Basic                               900                1379
1                  None                               885                1326
2              Advanced                               839                1294
3                Europe                                 1                   1


What is the average Performance_Rating of employees who completed each Training_Program_Name?
How does Employee_Training_Evaluation_Score correlate with Performance_Rating for employees who completed training?

In [ ]:
query_high_potential = """
SELECT
    tp.Training_Program_Name,
    AVG(e.Performance_Rating) AS Avg_Performance_Rating,
    AVG(e.Employee_Training_Evaluation_Score) AS Avg_Training_Evaluation_Score,
    COUNT(DISTINCT ete.Employee_ID) AS Participant_Count
FROM
    Training_Programs AS tp
JOIN
    Employee_Training_Enrollments AS ete ON tp.Training_Program_ID = ete.Training_Program_ID
JOIN
    Employees AS e ON ete.Employee_ID = e.Employee_ID
GROUP BY
    tp.Training_Program_Name
ORDER BY
    Avg_Performance_Rating DESC;
"""

high_potential_employees = pd.read_sql(query_high_potential, conn)
print(high_potential_employees)

  Training_Program_Name  Avg_Performance_Rating  \
0                Europe               13.000000   
1                  None               10.023810   
2                 Basic                9.915888   
3              Advanced                9.875762   

   Avg_Training_Evaluation_Score  Participant_Count  
0                      83.000000                  1  
1                      79.755900               1326  
2                      79.864845               1379  
3                      80.166921               1294  


Which employees meet the high-potential criteria (example thresholds: Performance_Rating > 10, Professional_Development_Hours > 50, Certifications > 2)?
Output fields: Employee_ID, Job_Title, Performance_Rating, Professional_Development_Hours, Certifications.


In [ ]:
query_high_potential = """
SELECT
    E.Employee_ID,
    E.Job_Title,
    E.Performance_Rating,
    SUM(ETE.Professional_Development_Hours) AS Total_Professional_Development_Hours,
    E.Certifications
FROM
    Employees AS E
JOIN
    Employee_Training_Enrollments AS ETE
    ON E.Employee_ID = ETE.Employee_ID
GROUP BY
    E.Employee_ID, E.Job_Title, E.Performance_Rating, E.Certifications
HAVING
    E.Performance_Rating > 10 AND
    SUM(ETE.Professional_Development_Hours) > 50 AND
    E.Certifications > 2
ORDER BY
    E.Performance_Rating DESC, Total_Professional_Development_Hours DESC;
"""

high_potential_employees = pd.read_sql(query_high_potential, conn)
print("High-Potential Employees:")
print(high_potential_employees)
print("\n" + "="*50 + "\n")

High-Potential Employees:
             Employee_ID   Job_Title  Performance_Rating  \
0           2024/IT/2369     Manager                20.0   
1        2024/Sales/1961     Manager                20.0   
2    2024/Marketing/0372  Consultant                19.0   
3      2024/Finance/2810  Consultant                19.0   
4           2024/IT/1436  Consultant                19.0   
..                   ...         ...                 ...   
603      2024/Sales/0058   Developer                11.0   
604      2024/Sales/0232     Analyst                11.0   
605      2024/Sales/1809     Analyst                11.0   
606      2024/Sales/3234  Consultant                11.0   
607      2024/Sales/3472     Manager                11.0   

     Total_Professional_Development_Hours Certifications  
0                                    54.0            PMP  
1                                    51.0   Google Cloud  
2                                   220.0            CFA  
3                

Which employees are at risk of resignation based on Employee_Job_Satisfaction_Score < threshold OR Work_Life_Balance_Rating < threshold?
Output fields: Employee_ID, Job_Title, Employee_Job_Satisfaction_Score, Work_Life_Balance_Rating, Employee_Resignation_Status.


In [ ]:
query_at_risk_resignation = """
SELECT
    Employee_ID,
    Job_Title,
    Employee_Job_Satisfaction_Score,
    Work_Life_Balance_Rating,
    Employee_Resignation_Status
FROM
    Employees
WHERE
    Employee_Job_Satisfaction_Score < 3 OR Work_Life_Balance_Rating < 2
ORDER BY
    Employee_Job_Satisfaction_Score ASC, Work_Life_Balance_Rating ASC;
"""

at_risk_employees = pd.read_sql(query_at_risk_resignation, conn)
print("Employees at Risk of Resignation:")
print(at_risk_employees)
print("\n" + "="*50 + "\n")

Employees at Risk of Resignation:
            Employee_ID   Job_Title  Employee_Job_Satisfaction_Score  \
0   2024/Marketing/3089  Consultant                                0   
1       2024/Sales/1985   Developer                                0   
2       2024/Sales/3411     Analyst                                1   
3          2024/HR/0006   Developer                                1   
4          2024/HR/1123     Analyst                                1   
..                  ...         ...                              ...   
64    2024/Finance/2320     Analyst                                9   
65         2024/HR/3091     Analyst                                9   
66         2024/IT/3958     Manager                                9   
67         2024/HR/0887     Manager                               10   
68      2024/Sales/3540     Manager                               10   

    Work_Life_Balance_Rating Employee_Resignation_Status  
0                          7              

Who are the direct reports, with Employee_ID, Job_Title, Performance_Rating, Salary, Employee_Resignation_Status?

In [ ]:
query = """
SELECT
    Employee_ID,
    UPPER(TRIM(Job_Title)) AS Job_Title,
    Performance_Rating,
    Salary,
    Employee_Resignation_Status
FROM Employees
WHERE Manager_ID = '2024/Sales/2005'
ORDER BY Performance_Rating DESC;
"""

# Execute and display results
direct_reports = pd.read_sql(query, conn)
direct_reports

,Employee_ID,Job_Title,Performance_Rating,Salary,Employee_Resignation_Status
0,2024/Marketing/3138,CONSULTANT,25.0,80501.0,No
1,2024/Finance/2880,ANALYST,11.0,79851.0,No


What is the average Performance_Rating and average Employee_Engagement_Score for this manager's team?
Output fields: Direct report list and team aggregates: Avg_Performance_Rating, Avg_Employee_Engagement_Score, Team_Size.


used a single manager id as given in question that say use 101 id

In [ ]:
query = """
SELECT
    Employee_ID,
    UPPER(TRIM(Job_Title)) AS Job_Title,
    Performance_Rating,
    Salary,
    Employee_Resignation_Status
FROM Employees
WHERE Manager_ID = '2024/Sales/2005'
ORDER BY Performance_Rating DESC;
"""

direct_reports = pd.read_sql(query, conn)
direct_reports

,Employee_ID,Job_Title,Performance_Rating,Salary,Employee_Resignation_Status
0,2024/Marketing/3138,CONSULTANT,25.0,80501.0,No
1,2024/Finance/2880,ANALYST,11.0,79851.0,No


What is the overall internship conversion rate (Internship_Conversion_Status = 'Converted')?


In [ ]:
query_overall_conversion = """
SELECT
    'Overall' AS "Group",
    CAST(SUM(CASE WHEN Internship_Conversion_Status = 'Converted' THEN 1 ELSE 0 END) AS REAL) * 100 / COUNT(*) AS Conversion_Rate,
    COUNT(*) AS Total_Interns,
    SUM(CASE WHEN Internship_Conversion_Status = 'Converted' THEN 1 ELSE 0 END) AS Converted_Count
FROM
    Internships;
"""

overall_conversion_rate = pd.read_sql(query_overall_conversion, conn)
print("Overall Internship Conversion Rate:")
print(overall_conversion_rate)
print("\n" + "="*50 + "\n")

Overall Internship Conversion Rate:
     Group  Conversion_Rate  Total_Interns  Converted_Count
0  Overall        48.705179           4016             1956




How do conversion rates vary by Internship_Learning_Outcomes and by Internship_Duration?


In [ ]:
query_learning_outcomes_conversion = """
SELECT
    Internship_Learning_Outcomes AS "Group",
    CAST(SUM(CASE WHEN Internship_Conversion_Status = 'Converted' THEN 1 ELSE 0 END) AS REAL) * 100 / COUNT(*) AS Conversion_Rate,
    COUNT(*) AS Total_Interns,
    SUM(CASE WHEN Internship_Conversion_Status = 'Converted' THEN 1 ELSE 0 END) AS Converted_Count
FROM
    Internships
WHERE
    Internship_Learning_Outcomes IS NOT NULL -- Exclude rows where learning outcome is not specified
GROUP BY
    Internship_Learning_Outcomes
ORDER BY
    Conversion_Rate DESC;
"""

learning_outcomes_conversion = pd.read_sql(query_learning_outcomes_conversion, conn)
print("Conversion Rates by Internship Learning Outcomes:")
print(learning_outcomes_conversion)
print("\n" + "="*50 + "\n")

Conversion Rates by Internship Learning Outcomes:
                Group  Conversion_Rate  Total_Interns  Converted_Count
0  Learning Outcome A        48.785714           1400              683
1  Learning Outcome C        48.782344           1314              641
2  Learning Outcome B        48.540707           1302              632




In [ ]:
query_duration_conversion = """
SELECT
    CAST(Internship_Duration AS TEXT) || ' months' AS "Group",
    CAST(SUM(CASE WHEN Internship_Conversion_Status = 'Converted' THEN 1 ELSE 0 END) AS REAL) * 100 / COUNT(*) AS Conversion_Rate,
    COUNT(*) AS Total_Interns,
    SUM(CASE WHEN Internship_Conversion_Status = 'Converted' THEN 1 ELSE 0 END) AS Converted_Count
FROM
    Internships
WHERE
    Internship_Duration IS NOT NULL -- Exclude rows where duration is not specified
GROUP BY
    Internship_Duration
ORDER BY
    Internship_Duration ASC;
"""

duration_conversion = pd.read_sql(query_duration_conversion, conn)
print("Conversion Rates by Internship Duration:")
print(duration_conversion)
print("\n" + "="*50 + "\n")

Conversion Rates by Internship Duration:
        Group  Conversion_Rate  Total_Interns  Converted_Count
0  1.0 months        49.839744            624              311
1  2.0 months        48.595506            712              346
2  3.0 months        49.280000            625              308
3  4.0 months        48.168250            737              355
4  5.0 months        48.137109            671              323
5  6.0 months        48.377125            647              313


